In [ ]:
!hf download airvlab/Grasp-Anything-pp --repo-type dataset --local-dir data/grasp-anything++/

Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0% 0/6 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/2.31k [00:00<?, ?B/s]         

Fetching 6 files:  17% 1/6 [00:00<00:01,  4.85it/s]
Reconstructing (incomplete total...):   0% 2.31k/4.79G [00:00<118:04:10, 11.3kB/s]
Reconstructing (incomplete total...):   0% 2.31k/6.34G [00:00<156:06:16, 11.3kB/s]
Reconstructing (incomplete total...):   0% 2.31k/15.4G [00:00<379:35:18, 11.3kB/s]
Reconstructing (incomplete total...):   0% 2.31k/15.4G [00:00<379:35:18, 11.3kB/s]
Reconstructing (incomplete total...):  10%

In [ ]:
from pathlib import Path
import pickle
import re
import shutil
import zipfile

DATA_DIR = Path("data/grasp-anything++")
SEEN_OBJ = "seen.obj"
UNSEEN_OBJ = "unseen.obj"
OUTPUT_DIR = Path("extracted_matched_pp")

ZIP_FILES = [
    "grasp_instructions.zip",
    "grasp_label_positive.zip",
    "grasp_label_negative.zip",
    "part_mask.zip",
]

# "union"        -> keep items in seen.obj OR unseen.obj
# "intersection" -> keep only items in BOTH seen.obj AND unseen.obj
MATCH_MODE = "union"

# Set True to remove each .zip right after extracting from it (saves disk space)
DELETE_ZIP_AFTER_EXTRACT = True
CHUNK_SIZE = 1024 * 1024  # 1 MB


def load_split_ids(obj_path: str) -> tuple[set[str], set[str]]:
  """Return (sample_ids like '<hash>_<idx>', base_hashes like '<hash>')."""
  try:
    with open(obj_path, "rb") as f:
      data = pickle.load(f)
    items = data.keys() if isinstance(data, dict) else data
  except Exception:
    with open(obj_path, "r", encoding="utf-8") as f:
      items = [line.strip() for line in f if line.strip()]

  sample_ids = set()
  base_hashes = set()
  for item in items:
    stem = Path(str(item).strip()).stem
    if not stem:
      continue
    sample_ids.add(stem)
    base_hashes.add(re.sub(r"_\d+$", "", stem))
  return sample_ids, base_hashes


def is_matching_member(
    stem: str, target_samples: set[str], target_hashes: set[str]
) -> bool:
  """Match scene-level (<hash>), sample-level (<hash>_<idx>), or part-level (<hash>_<idx>_<part>)."""
  if stem in target_samples or stem in target_hashes:
    return True
  parent_sample = re.sub(r"_\d+$", "", stem)
  return parent_sample in target_samples


seen_samples, seen_hashes = load_split_ids(SEEN_OBJ)
unseen_samples, unseen_hashes = load_split_ids(UNSEEN_OBJ)

if MATCH_MODE == "intersection":
  target_samples = seen_samples & unseen_samples
  target_hashes = seen_hashes & unseen_hashes
else:
  target_samples = seen_samples | unseen_samples
  target_hashes = seen_hashes | unseen_hashes

print(
    f"Loaded {len(target_samples)} sample IDs and {len(target_hashes)} scene"
    f" hashes ({MATCH_MODE})."
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for zip_name in ZIP_FILES:
  zip_path = DATA_DIR / zip_name
  if not zip_path.exists():
    print(f"[Skip] {zip_path} not found.")
    continue

  sub_out = OUTPUT_DIR / zip_path.stem
  sub_out.mkdir(parents=True, exist_ok=True)
  extracted = 0

  print(f"Extracting matches from {zip_path}...")
  with zipfile.ZipFile(zip_path, "r") as zf:
    for member in zf.infolist():
      if member.is_dir():
        continue

      member_path = Path(member.filename)
      if ".." in member_path.parts:
        continue

      if not is_matching_member(
          member_path.stem, target_samples, target_hashes
      ):
        continue

      target_path = (sub_out / member_path.name).resolve()
      if not target_path.is_relative_to(sub_out.resolve()):
        continue

      with zf.open(member) as src, open(target_path, "wb") as dst:
        shutil.copyfileobj(src, dst, length=CHUNK_SIZE)
      extracted += 1

  print(f"  Extracted {extracted} files to {sub_out}")

  if DELETE_ZIP_AFTER_EXTRACT:
    zip_path.unlink()
    print(f"  Deleted {zip_path} to free disk space.")

Loaded 14746 sample IDs and 11779 scene hashes (union).
Extracting matches from data/grasp-anything++/grasp_instructions.zip...
  Extracted 29222 files to extracted_matched_pp/grasp_instructions
  Deleted data/grasp-anything++/grasp_instructions.zip to free disk space.
Extracting matches from data/grasp-anything++/grasp_label_positive.zip...
  Extracted 29222 files to extracted_matched_pp/grasp_label_positive
  Deleted data/grasp-anything++/grasp_label_positive.zip to free disk space.
Extracting matches from data/grasp-anything++/grasp_label_negative.zip...
  Extracted 29222 files to extracted_matched_pp/grasp_label_negative
  Deleted data/grasp-anything++/grasp_label_negative.zip to free disk space.
Extracting matches from data/grasp-anything++/part_mask.zip...
  Extracted 29222 files to extracted_matched_pp/part_mask
  Deleted data/grasp-anything++/part_mask.zip to free disk space.


In [ ]:
import subprocess
from google.colab import files

BASE_DIR = Path("extracted_matched_pp")
DELETE_FOLDER_AFTER_ZIP = (
    True  # Frees disk space before zipping the next folder
)

for subfolder in sorted(p for p in BASE_DIR.iterdir() if p.is_dir()):
  zip_name = f"{subfolder.name}.zip"
  print(f"Zipping {subfolder} -> {zip_name}...")

  # -q suppresses per-file spam in Colab output
  subprocess.run(["zip", "-rq", zip_name, str(subfolder)], check=True)

  if DELETE_FOLDER_AFTER_ZIP:
    shutil.rmtree(subfolder)

  print(f"Downloading {zip_name}...")
  files.download(zip_name)

Zipping extracted_matched_pp/grasp_instructions -> grasp_instructions.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Zipping extracted_matched_pp/grasp_label_negative -> grasp_label_negative.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (2 outputs left out)

<IPython.core.display.Javascript object>

Zipping extracted_matched_pp/part_mask -> part_mask.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>